In [ ]:
%pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv pandas pyarrow

In [ ]:
import os
os.makedirs('/tmp/ouro2', exist_ok=True)

In [ ]:
%%writefile /tmp/ouro2/config.py
"""Single configuration surface for the V2 harness.

One frozen dataclass, populated from at most eight OURO2_* environment
variables. There is deliberately no other configuration mechanism: every
component receives the same Config instance, so a budget or model knob can
never disagree between call sites (the V1 failure this design replaces).
"""
from __future__ import annotations

import os
from dataclasses import dataclass


@dataclass(frozen=True)
class Config:
    max_actions: int = 320
    disable_model: bool = True
    model_path: str = ""
    model_backend: str = "ollama"  # "ollama" | "transformers"
    model_max_calls: int = 24
    time_budget_s: float = 600.0  # per-game CPU/wall budget for thinking
    trace_path: str = ""
    node_cap: int = 20000  # planner search-node ceiling

    @classmethod
    def from_env(cls) -> "Config":
        env = os.environ.get
        return cls(
            max_actions=int(env("OURO2_MAX_ACTIONS", "320")),
            disable_model=env("OURO2_DISABLE_MODEL", "1") not in ("", "0"),
            model_path=env("OURO2_MODEL_PATH", ""),
            model_backend=env("OURO2_MODEL_BACKEND", "ollama"),
            model_max_calls=int(env("OURO2_MODEL_MAX_CALLS", "24")),
            time_budget_s=float(env("OURO2_TIME_BUDGET_S", "600")),
            trace_path=env("OURO2_TRACE_PATH", ""),
            node_cap=int(env("OURO2_NODE_CAP", "20000")),
        )


In [ ]:
%%writefile /tmp/ouro2/grid.py
"""Grid primitives.

A grid is an immutable ``bytes`` of length 4096 (64x64, row-major, colors
0-15). ARC-AGI-3 always serves 64x64 frames; fixing the shape keeps every
hot path allocation-free and hashable. Cell (x, y) lives at index y*64+x.
"""
from __future__ import annotations

import hashlib
from collections import Counter, deque
from dataclasses import dataclass
from functools import lru_cache

SIZE = 64
CELLS = SIZE * SIZE

Grid = bytes


def to_grid(frame_stack: list[list[list[int]]] | None) -> Grid | None:
    """Last grid of a FrameData.frame stack, or None for empty/burned frames.

    Duck-typed (len/int casts, no truthiness): the real engine returns numpy
    arrays, whose truth value raises.
    """
    if frame_stack is None or len(frame_stack) == 0:
        return None
    rows = frame_stack[-1]
    if rows is None or len(rows) == 0:
        return None
    flat = bytearray(CELLS)
    for y in range(min(len(rows), SIZE)):
        row = rows[y]
        for x in range(min(len(row), SIZE)):
            flat[y * SIZE + x] = int(row[x]) & 0x0F
    return bytes(flat)


def from_rows(rows: list[list[int]]) -> Grid:
    """Build a grid from (possibly smaller) row data, zero-padded to 64x64."""
    return to_grid([rows]) or bytes(CELLS)


def cell(g: Grid, x: int, y: int) -> int:
    return g[y * SIZE + x]


def grid_key(g: Grid) -> str:
    """Stable short key (safe across processes, unlike hash(bytes))."""
    return hashlib.blake2b(g, digest_size=8).hexdigest()


def masked_key(g: Grid, volatile: frozenset[int], keep_color: int | None = None) -> str:
    """State key with volatile cells zeroed (animation-proof identity).

    Cells currently holding ``keep_color`` (the avatar) are never zeroed:
    the avatar's position IS the state, even when it stands on a cell the
    volatility mask covers (e.g. inside a patroller's corridor).
    """
    if not volatile:
        return grid_key(g)
    flat = bytearray(g)
    for i in volatile:
        if keep_color is None or flat[i] != keep_color:
            flat[i] = 0
    return hashlib.blake2b(bytes(flat), digest_size=8).hexdigest()


def diff(a: Grid, b: Grid) -> list[tuple[int, int, int, int]]:
    """Changed cells as (x, y, old, new)."""
    out = []
    for i in range(CELLS):
        if a[i] != b[i]:
            out.append((i % SIZE, i // SIZE, a[i], b[i]))
    return out


def color_counts(g: Grid) -> dict[int, int]:
    return dict(Counter(g))


@lru_cache(maxsize=8192)
def most_common_color(g: Grid) -> int:
    return Counter(g).most_common(1)[0][0]


@dataclass(frozen=True)
class Obj:
    color: int
    cells: frozenset[tuple[int, int]]
    bbox: tuple[int, int, int, int]  # x0, y0, x1, y1 inclusive
    centroid: tuple[int, int]
    shape_hash: str

    @property
    def size(self) -> int:
        return len(self.cells)

    @property
    def width(self) -> int:
        return self.bbox[2] - self.bbox[0] + 1

    @property
    def height(self) -> int:
        return self.bbox[3] - self.bbox[1] + 1


def _shape_hash(color: int, cells: list[tuple[int, int]], x0: int, y0: int) -> str:
    normalized = sorted((x - x0, y - y0) for x, y in cells)
    payload = bytes([color]) + b"".join(bytes((x, y)) for x, y in normalized)
    return hashlib.blake2b(payload, digest_size=6).hexdigest()


def components(
    g: Grid,
    colors: set[int] | None = None,
    conn: int = 4,
    background: int | None = None,
) -> list[Obj]:
    """Same-color connected components, largest first.

    ``colors`` restricts which colors to segment; ``background`` (default:
    most common color) is skipped entirely.
    """
    bg = most_common_color(g) if background is None else background
    seen = bytearray(CELLS)
    offsets4 = ((1, 0), (-1, 0), (0, 1), (0, -1))
    offsets8 = offsets4 + ((1, 1), (1, -1), (-1, 1), (-1, -1))
    offsets = offsets8 if conn == 8 else offsets4
    out: list[Obj] = []
    for start in range(CELLS):
        if seen[start]:
            continue
        color = g[start]
        seen[start] = 1
        if color == bg or (colors is not None and color not in colors):
            continue
        cells = [(start % SIZE, start // SIZE)]
        queue = deque(cells)
        while queue:
            cx, cy = queue.popleft()
            for dx, dy in offsets:
                nx, ny = cx + dx, cy + dy
                if 0 <= nx < SIZE and 0 <= ny < SIZE:
                    idx = ny * SIZE + nx
                    if not seen[idx] and g[idx] == color:
                        seen[idx] = 1
                        cells.append((nx, ny))
                        queue.append((nx, ny))
        xs = [c[0] for c in cells]
        ys = [c[1] for c in cells]
        x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)
        out.append(
            Obj(
                color=color,
                cells=frozenset(cells),
                bbox=(x0, y0, x1, y1),
                centroid=(sum(xs) // len(xs), sum(ys) // len(ys)),
                shape_hash=_shape_hash(color, cells, x0, y0),
            )
        )
    out.sort(key=lambda o: (-o.size, o.color, o.bbox))
    return out


def apply_cells(g: Grid, cells: list[tuple[int, int, int]]) -> Grid:
    """Return a copy of ``g`` with (x, y, color) writes applied."""
    if not cells:
        return g
    flat = bytearray(g)
    for x, y, color in cells:
        flat[y * SIZE + x] = color & 0x0F
    return bytes(flat)


In [ ]:
%%writefile /tmp/ouro2/timeline.py
"""Append-only interaction record.

The timeline is the ground truth every rule hypothesis is tested against
(Schema's "record" stage). Hypotheses are revisable; the observations and
actions here are not. Segmentation into plays (full resets) and levels is
derived, never stored twice.
"""
from __future__ import annotations

from dataclasses import dataclass, field

from .grid import Grid


@dataclass(frozen=True)
class ActionSpec:
    action: int  # 0=RESET, 1-5, 6=click, 7
    x: int | None = None
    y: int | None = None
    source: str = ""
    reason: str = ""

    def is_reset(self) -> bool:
        return self.action == 0

    def key(self) -> tuple[int, int | None, int | None]:
        return (self.action, self.x, self.y)


RESET = ActionSpec(0, source="reset")


@dataclass(frozen=True)
class Transition:
    index: int
    level: int  # level the action was taken in (levels_completed before)
    before: Grid | None
    action: ActionSpec
    after: Grid | None  # None for burned actions (empty frame stack)
    state_after: str  # NOT_FINISHED | WIN | GAME_OVER | NOT_PLAYED
    levels_before: int
    levels_after: int
    full_reset: bool

    @property
    def level_up(self) -> bool:
        return self.levels_after > self.levels_before

    @property
    def burned(self) -> bool:
        return self.after is None


@dataclass
class LevelSegment:
    play: int
    level: int
    transitions: list[Transition] = field(default_factory=list)

    @property
    def completed(self) -> bool:
        return bool(self.transitions) and self.transitions[-1].level_up


class Timeline:
    def __init__(self) -> None:
        self.transitions: list[Transition] = []

    def __len__(self) -> int:
        return len(self.transitions)

    def append(
        self,
        before: Grid | None,
        action: ActionSpec,
        after: Grid | None,
        state_after: str,
        levels_before: int,
        levels_after: int,
        full_reset: bool = False,
    ) -> Transition:
        t = Transition(
            index=len(self.transitions),
            level=levels_before,
            before=before,
            action=action,
            after=after,
            state_after=state_after,
            levels_before=levels_before,
            levels_after=levels_after,
            full_reset=full_reset,
        )
        self.transitions.append(t)
        return t

    def plays(self) -> list[list[Transition]]:
        """Split on full resets; the reset transition starts the new play."""
        out: list[list[Transition]] = [[]]
        for t in self.transitions:
            if t.full_reset and out[-1]:
                out.append([])
            out[-1].append(t)
        return out if out[0] else out[1:]

    def levels(self) -> list[LevelSegment]:
        segments: list[LevelSegment] = []
        for play_idx, play in enumerate(self.plays()):
            current: LevelSegment | None = None
            for t in play:
                if current is None or t.level != current.level:
                    current = LevelSegment(play=play_idx, level=t.level)
                    segments.append(current)
                current.transitions.append(t)
        return segments

    def current_level_transitions(self) -> list[Transition]:
        if not self.transitions:
            return []
        level = self.transitions[-1].levels_after
        plays = self.plays()
        return [t for t in plays[-1] if t.level == level]

    def winning_macro(self) -> list[list[ActionSpec]] | None:
        """Per-level action sequences from the play that reached WIN.

        Includes every action taken while on that level (RESETs and all) —
        the speedrun replays this verbatim; compression is the planner's job.
        """
        for play in self.plays():
            if not any(t.state_after == "WIN" for t in play):
                continue
            per_level: dict[int, list[ActionSpec]] = {}
            for t in play:
                per_level.setdefault(t.level, []).append(t.action)
            return [per_level[level] for level in sorted(per_level)]
        return None


In [ ]:
%%writefile /tmp/ouro2/rules.py
"""The world model: rules as data, run by a fixed interpreter.

Rules are induced by the CPU (induce.py) and never authored by the LLM, so
this interpreter is trusted code and needs no sandbox. It is total by
construction: one pass over the rules per action, push chains bounded by
the grid width, no recursion.

State is the grid itself plus consumed-item counters — exact, hashable,
and cheap enough at the planner's node cap. Painting semantics: cells the
avatar leaves are restored to the background color (games that stack items
under the avatar are represented via ``consumes``).
"""
from __future__ import annotations

from dataclasses import dataclass, field
from functools import lru_cache

from .grid import CELLS, SIZE, Grid, components, most_common_color


@dataclass(frozen=True)
class Binding:
    """How raw pixels are read as entities (the representation)."""

    avatar_color: int | None = None
    avatar_extra: frozenset[int] = frozenset()  # companion sprite colors
    background: int | None = None
    conn: int = 4

    def bg(self, g: Grid) -> int:
        return most_common_color(g) if self.background is None else self.background


@dataclass(frozen=True)
class State:
    grid: Grid
    counters: tuple[tuple[int, int], ...] = ()  # (color, consumed count), sorted
    status: str = "NOT_FINISHED"  # NOT_FINISHED | GAME_OVER

    def counter(self, color: int) -> int:
        for c, n in self.counters:
            if c == color:
                return n
        return 0

    def with_counter(self, color: int, added: int) -> "State":
        counts = dict(self.counters)
        counts[color] = counts.get(color, 0) + added
        return State(self.grid, tuple(sorted(counts.items())), self.status)


@dataclass(frozen=True)
class MoveRule:
    """Avatar movement. Subsumes push (pushable), collect (consumes) and
    death-on-contact (on_block="die")."""

    deltas: tuple[tuple[int, tuple[int, int]], ...]  # (action, (dx, dy))
    blockers: frozenset[int] = frozenset()
    pushable: frozenset[int] = frozenset()
    consumes: frozenset[int] = frozenset()
    on_block: str = "stay"  # stay | die
    slide: bool = False  # repeat the step until blocked (ice)
    floor: int | None = None  # color vacated cells restore to (default: bg)

    kind: str = field(default="move", init=False)

    def delta_for(self, action: int) -> tuple[int, int] | None:
        for a, d in self.deltas:
            if a == action:
                return d
        return None


@dataclass(frozen=True)
class ClickRule:
    """ACTION6 recoloring. scope: cell | object | color."""

    scope: str
    mapping: tuple[tuple[int, int], ...]  # (from_color, to_color)

    kind: str = field(default="click_effect", init=False)

    def to_color(self, color: int) -> int | None:
        for src, dst in self.mapping:
            if src == color:
                return dst
        return None


@dataclass(frozen=True)
class TickRule:
    """Passive per-action translation of all objects of one color."""

    color: int
    delta: tuple[int, int]

    kind: str = field(default="tick_move", init=False)


@dataclass(frozen=True)
class HazardRule:
    """Avatar sharing a cell edge with any of these colors ends the game."""

    colors: frozenset[int]

    kind: str = field(default="hazard", init=False)


Rule = MoveRule | ClickRule | TickRule | HazardRule
RuleSet = tuple[Rule, ...]


@dataclass(frozen=True)
class Goal:
    kind: str  # reach_color | clear_color | counter_eq
    color: int
    count: int = 0


def avatar_cells(g: Grid, binding: Binding) -> frozenset[tuple[int, int]]:
    """The avatar patch: the largest avatar-color component, grown over the
    companion colors (multi-color sprites move as one rigid patch)."""
    if binding.avatar_color is None:
        return frozenset()
    return _avatar_cells_cached(
        g, binding.avatar_color, binding.avatar_extra, binding.conn
    )


@lru_cache(maxsize=8192)
def _avatar_cells_cached(
    g: Grid, color: int, extra: frozenset[int], conn: int
) -> frozenset[tuple[int, int]]:
    # Collect candidate cells in one scan, flood-fill among them, and keep
    # EVERY component containing at least one primary-color cell: games run
    # multiple simultaneously-controlled avatars (m0r0's twin sprites), and
    # companion-color parts attach per component.
    union = {color} | set(extra)
    cells = {(i % SIZE, i // SIZE) for i, c in enumerate(g) if c in union}
    if not cells:
        return frozenset()
    offsets = (
        ((1, 0), (-1, 0), (0, 1), (0, -1))
        if conn == 4
        else ((1, 0), (-1, 0), (0, 1), (0, -1), (1, 1), (1, -1), (-1, 1), (-1, -1))
    )
    out: set[tuple[int, int]] = set()
    remaining = set(cells)
    while remaining:
        seed = remaining.pop()
        group = {seed}
        frontier = [seed]
        while frontier:
            x, y = frontier.pop()
            for dx, dy in offsets:
                p = (x + dx, y + dy)
                if p in remaining:
                    remaining.remove(p)
                    group.add(p)
                    frontier.append(p)
        if any(g[y * SIZE + x] == color for x, y in group):
            out |= group
    return frozenset(out)


def extract_state(g: Grid) -> State:
    return State(grid=g)


def _move_once(
    flat: bytearray,
    cells: frozenset[tuple[int, int]],
    dx: int,
    dy: int,
    rule: MoveRule,
    bg: int,
    avatar_color: int,
) -> tuple[frozenset[tuple[int, int]], list[int], bool]:
    """One movement step. Returns (new cells, consumed colors, moved?)."""
    restore = bg if rule.floor is None else rule.floor
    walkable = {bg, restore}
    dest = {(x + dx, y + dy) for x, y in cells}
    if any(not (0 <= x < SIZE and 0 <= y < SIZE) for x, y in dest):
        return cells, [], False
    entered = dest - cells
    entered_colors = {flat[y * SIZE + x] for x, y in entered}
    push_groups: list[frozenset[tuple[int, int]]] = []
    frontier = {p for p in entered if flat[p[1] * SIZE + p[0]] in rule.pushable}
    pushed: set[tuple[int, int]] = set()
    steps = 0
    while frontier and steps <= SIZE:
        steps += 1
        group = frozenset(frontier)
        pushed |= group
        push_groups.append(group)
        nxt = {(x + dx, y + dy) for x, y in group} - pushed - cells
        blocked = False
        for x, y in nxt:
            if not (0 <= x < SIZE and 0 <= y < SIZE):
                blocked = True
                break
            c = flat[y * SIZE + x]
            if c in rule.blockers or (
                c not in rule.pushable and c not in walkable and c not in rule.consumes
            ):
                blocked = True
                break
        if blocked:
            return cells, [], False
        frontier = {p for p in nxt if flat[p[1] * SIZE + p[0]] in rule.pushable}
    if any(
        c in rule.blockers
        or (c not in walkable and c not in rule.consumes and c not in rule.pushable)
        for c in entered_colors
    ):
        return cells, [], False
    consumed = [
        flat[y * SIZE + x]
        for x, y in entered
        if flat[y * SIZE + x] in rule.consumes
    ]
    patch_colors = {(x, y): flat[y * SIZE + x] for x, y in cells}
    # Move pushed groups first (furthest groups were appended later).
    for group in reversed(push_groups):
        colors = {(x, y): flat[y * SIZE + x] for x, y in group}
        for x, y in group:
            flat[y * SIZE + x] = bg
        for (x, y), c in colors.items():
            flat[(y + dy) * SIZE + (x + dx)] = c
    for x, y in cells:
        flat[y * SIZE + x] = restore
    for (x, y), c in patch_colors.items():
        flat[(y + dy) * SIZE + (x + dx)] = c
    return frozenset(dest), consumed, True


def step(state: State, action_key: tuple[int, int | None, int | None],
         rules: RuleSet, binding: Binding) -> tuple[State, str]:
    """Apply one action to the model state. Returns (next state, outcome).

    Outcomes: moved | blocked | died | clicked | noop — plus status change
    to GAME_OVER on hazard contact or on_block="die".
    """
    if state.status == "GAME_OVER":
        return state, "noop"
    action, cx, cy = action_key
    flat = bytearray(state.grid)
    bg = binding.bg(state.grid)
    outcome = "noop"
    consumed: list[int] = []

    if action == 6 and cx is not None and cy is not None:
        target = flat[cy * SIZE + cx]
        for rule in rules:
            if isinstance(rule, ClickRule):
                dst = rule.to_color(target)
                if dst is None:
                    continue
                if rule.scope == "cell":
                    flat[cy * SIZE + cx] = dst
                elif rule.scope == "object":
                    for obj in components(bytes(flat), colors={target}, background=bg):
                        if (cx, cy) in obj.cells:
                            for x, y in obj.cells:
                                flat[y * SIZE + x] = dst
                            break
                else:  # color
                    for i in range(CELLS):
                        if flat[i] == target:
                            flat[i] = dst
                outcome = "clicked"
                break

    move_rule = next((r for r in rules if isinstance(r, MoveRule)), None)
    if move_rule is not None and binding.avatar_color is not None:
        delta = move_rule.delta_for(action)
        if delta is not None:
            cells = avatar_cells(bytes(flat), binding)
            if cells:
                moved_any = False
                while True:
                    cells2, eaten, moved = _move_once(
                        flat, cells, delta[0], delta[1], move_rule, bg,
                        binding.avatar_color,
                    )
                    consumed.extend(eaten)
                    if not moved:
                        break
                    moved_any = True
                    cells = cells2
                    if not move_rule.slide:
                        break
                if moved_any:
                    outcome = "moved"
                elif outcome == "noop":
                    if move_rule.on_block == "die":
                        return State(bytes(flat), state.counters, "GAME_OVER"), "died"
                    outcome = "blocked"

    for rule in rules:
        if isinstance(rule, TickRule):
            dx, dy = rule.delta
            objs = components(bytes(flat), colors={rule.color}, background=bg)
            for obj in objs:
                dest = {(x + dx, y + dy) for x, y in obj.cells}
                if any(not (0 <= x < SIZE and 0 <= y < SIZE) for x, y in dest):
                    continue
                if any(
                    flat[y * SIZE + x] not in (bg, rule.color)
                    for x, y in dest - obj.cells
                ):
                    continue
                for x, y in obj.cells:
                    flat[y * SIZE + x] = bg
                for x, y in dest:
                    flat[y * SIZE + x] = rule.color

    next_state = State(bytes(flat), state.counters, state.status)
    for color in consumed:
        next_state = next_state.with_counter(color, 1)

    hazard = next((r for r in rules if isinstance(r, HazardRule)), None)
    if hazard is not None and binding.avatar_color is not None:
        cells = avatar_cells(next_state.grid, binding)
        g = next_state.grid
        for x, y in cells:
            for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                if 0 <= nx < SIZE and 0 <= ny < SIZE and g[ny * SIZE + nx] in hazard.colors:
                    return State(g, next_state.counters, "GAME_OVER"), "died"

    return next_state, outcome


def is_goal(state: State, goal: Goal, binding: Binding) -> bool:
    g = state.grid
    if goal.kind == "clear_color":
        return goal.color not in g
    if goal.kind == "counter_eq":
        return state.counter(goal.color) >= goal.count
    if goal.kind == "reach_color":
        if goal.color not in g:
            return True  # consumed/painted over the last target cell
        for x, y in avatar_cells(g, binding):
            for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                if 0 <= nx < SIZE and 0 <= ny < SIZE and g[ny * SIZE + nx] == goal.color:
                    return True
        return False
    return False


In [ ]:
%%writefile /tmp/ouro2/induce.py
"""CPU rule induction: exact transition diffs -> parameterized rules.

The 4B never authors rules; it may only rank the candidates generated
here. Candidate generation is evidence-directed: a rigid translation of
color c by (dx, dy) admits only a move/tick rule with that delta; a
localized recolor admits only a click rule with that mapping. Evaluation
replays the whole timeline through the interpreter (Schema's
run_backtest): certification and prequential scoring are the same pass.
"""
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass, field

from .grid import Grid, components, diff, most_common_color
from .rules import (
    Binding,
    ClickRule,
    Goal,
    HazardRule,
    MoveRule,
    Rule,
    RuleSet,
    State,
    TickRule,
    avatar_cells,
    is_goal,
    step,
)
from .timeline import Timeline, Transition

SIMPLE_MOVE_ACTIONS = (1, 2, 3, 4)


def _cells_of(g: Grid, color: int) -> frozenset[tuple[int, int]]:
    return frozenset((i % 64, i // 64) for i, c in enumerate(g) if c == color)


def translated(before: Grid, after: Grid, color: int) -> tuple[int, int] | None:
    """(dx, dy) if every cell of ``color`` rigidly translated, else None.

    A rigid translation maps the lexicographic minimum of the set to the
    minimum of the image, so exactly one candidate delta needs checking —
    O(n), which matters because callers probe every changed color
    (including ~4000-cell backgrounds).
    """
    a = _cells_of(before, color)
    b = _cells_of(after, color)
    if not a or len(a) != len(b) or a == b or len(a) > 512:
        return None
    ax, ay = min(a)
    bx, by = min(b)
    dx, dy = bx - ax, by - ay
    if (dx, dy) != (0, 0) and {(x + dx, y + dy) for x, y in a} == b:
        return (dx, dy)
    return None


# ---------------------------------------------------------------------------
# Volatility mask: cells that keep changing regardless of what the rules
# explain (energy bars, timers, animations). Excluded from model comparison,
# per-step verification and state keys — Schema's observation that a model
# is "right" when mispredictions collapse, made cell-local.


def volatile_cells(timeline: Timeline, sample: int = 200) -> frozenset[int]:
    counts = [0] * 4096
    far_click = [0] * 4096
    n = 0
    for t in timeline.transitions[-sample:]:
        if t.before is None or t.after is None or t.action.is_reset() or t.level_up:
            continue
        n += 1
        is_click = t.action.action == 6 and t.action.x is not None
        for x, y, _, _ in diff(t.before, t.after):
            counts[y * 64 + x] += 1
            if is_click and max(abs(x - t.action.x), abs(y - t.action.y)) > 12:
                # Changed although the click was far away: action-independent
                # (attempt counters, progress strips) — HUD, not gameplay.
                far_click[y * 64 + x] += 1
    if n < 12:
        return frozenset()
    # Only near-every-transition churn qualifies (energy bars, timers,
    # animations). Gameplay cells an oscillating walk revisits stay firmly
    # below this — masking them would blind the executor where it matters.
    threshold = max(8, int(0.45 * n))
    out = {i for i, c in enumerate(counts) if c >= threshold}
    out |= {i for i, c in enumerate(far_click) if c >= 2}
    return frozenset(out)


def masked(
    g: Grid,
    volatile: frozenset[int],
    depleting: frozenset[int] = frozenset(),
    keep: int | None = None,
) -> Grid:
    """Zero volatile cells and depleting-color cells — except cells holding
    ``keep`` (the avatar): its position is the state."""
    if not volatile and not depleting:
        return g
    flat = bytearray(g)
    for i in volatile:
        if keep is None or flat[i] != keep:
            flat[i] = 0
    if depleting:
        for i in range(4096):
            if flat[i] in depleting and flat[i] != keep:
                flat[i] = 0
    return bytes(flat)


def masked_eq(
    a: Grid,
    b: Grid,
    volatile: frozenset[int],
    depleting: frozenset[int],
    keep: int | None = None,
) -> bool:
    """Equality under the identity masks, with depleting colors masked by
    UNION of both grids' positions: a freshly drained bar cell holds the
    depleting color in the stale grid but the revealed floor in the real
    one — both sides of that index must be ignored."""
    if a == b:
        return True
    for i in range(4096):
        av, bv = a[i], b[i]
        if av == bv:
            continue
        if i in volatile and av != keep and bv != keep:
            continue
        if (av in depleting or bv in depleting) and av != keep and bv != keep:
            continue
        return False
    return True


def depleting_colors(timeline: Timeline) -> frozenset[int]:
    """Colors whose cell count only ever falls within a level (energy bars,
    progress strips). They deplete on every action — unpredictable by any
    mechanic rule and poisonous to state identity — so they are masked at
    the COLOR level (their cells move as the bar drains, so per-cell
    frequency masking cannot see them)."""
    from collections import Counter as C

    decreases: C = C()
    increases: C = C()
    for seg in timeline.levels():
        prev: dict[int, int] | None = None
        for t in seg.transitions:
            if t.action.is_reset():
                prev = None  # a reset refills timers: break the count chain,
                continue      # or the refill masquerades as an "increase"
            if t.before is None or t.after is None:
                continue
            counts = C(t.after)
            if prev is not None:
                for color in set(prev) | set(counts):
                    d = counts.get(color, 0) - prev.get(color, 0)
                    if d < 0:
                        decreases[color] += 1
                    elif d > 0:
                        increases[color] += 1
            prev = dict(counts)
    return frozenset(
        c for c, n in decreases.items() if n >= 5 and increases.get(c, 0) == 0
    )  # depleting only: grow-only colors are often GAMEPLAY (paint); their
    # counter cells are caught cell-level via far-click volatility instead


# ---------------------------------------------------------------------------
# Binding (representation) induction


def _direction(delta: tuple[int, int]) -> tuple[int, int]:
    dx, dy = delta
    return ((dx > 0) - (dx < 0), (dy > 0) - (dy < 0))


def _consistent_majorities(by_action: dict) -> dict:
    """Per action, the majority DIRECTION — kept only when it dominates.

    Judged on direction rather than exact delta: a sliding avatar keeps its
    direction while its per-move distance varies with obstacles. A
    phase-correlated ticker still splits ~50/50 and must not qualify."""
    out = {}
    for action, counter in by_action.items():
        dirs = Counter()
        for delta, n in counter.items():
            dirs[_direction(delta)] += n
        direction, n = dirs.most_common(1)[0]
        total = sum(dirs.values())
        if total >= 2 and n / total >= 0.7:
            out[action] = (direction, n)
    return out


def _axis_jump(delta: tuple[int, int]) -> bool:
    """A plausible move delta: along one axis, 1-8 cells (ls20's avatar
    jumps 5 cells per press)."""
    dx, dy = delta
    return (dx == 0) != (dy == 0) and abs(dx) + abs(dy) <= 8


def rebind(timeline: Timeline, prior: Binding | None = None) -> Binding:
    """Pick the avatar color: the color whose translation is a FUNCTION of
    the action. A passive ticker moves identically whatever you press; the
    controlled object answers different actions with different deltas.
    This is the representation-revision entry point — called fresh on every
    (re)induction."""
    per_color: dict[int, dict[int, Counter]] = defaultdict(lambda: defaultdict(Counter))
    for t in _informative(timeline):
        if t.before is None or t.after is None or t.action.action not in SIMPLE_MOVE_ACTIONS:
            continue
        changed = diff(t.before, t.after)
        if not changed:
            continue
        bg = most_common_color(t.before)
        for color in {old for _, _, old, _ in changed}:
            if color == bg:
                continue  # backgrounds don't move; they get moved through
            delta = translated(t.before, t.after, color)
            if delta is not None and _axis_jump(delta):
                per_color[color][t.action.action][delta] += 1
    best_color = None
    best_score = 0
    for color, by_action in per_color.items():
        majority = _consistent_majorities(by_action)
        distinct = {delta for delta, _ in majority.values()}
        if len(majority) < 1 or (len(majority) >= 2 and len(distinct) < 2):
            continue  # constant delta across actions: a ticker, not an avatar
        score = sum(n for _, n in majority.values())
        if score > best_score:
            best_color, best_score = color, score
    # Loose pass (representation revision): multi-color sprites never
    # translate rigidly per color — vote on CENTROID displacement with
    # size tolerance instead (ls20's block with a mutating indicator).
    strict_winner = best_color
    per_color = defaultdict(lambda: defaultdict(Counter))
    for t in _informative(timeline):
        if t.before is None or t.after is None or t.action.action not in SIMPLE_MOVE_ACTIONS:
            continue
        bg = most_common_color(t.before)
        for color in {old for _, _, old, _ in diff(t.before, t.after)}:
            if color == bg:
                continue
            a = _cells_of(t.before, color)
            b = _cells_of(t.after, color)
            if not a or not b or len(a) > 512 or abs(len(a) - len(b)) > max(2, len(a) // 4):
                continue
            ax = sum(x for x, _ in a) / len(a)
            ay = sum(y for _, y in a) / len(a)
            bx = sum(x for x, _ in b) / len(b)
            by = sum(y for _, y in b) / len(b)
            delta = (round(bx - ax), round(by - ay))
            if _axis_jump(delta):
                per_color[color][t.action.action][delta] += 1
    for color, by_action in per_color.items():
        majority = _consistent_majorities(by_action)
        distinct = {delta for delta, _ in majority.values()}
        if len(majority) < 1 or (len(majority) >= 2 and len(distinct) < 2):
            continue
        score = sum(n for _, n in majority.values())
        if strict_winner is None and score > best_score:
            best_color, best_score = color, score
    if best_color is None:
        # Sticky binding: the window slides over phases where the avatar
        # stops moving (votes evaporate), but a binding is a durable fact —
        # keep the prior unless fresh evidence crowns a new winner.
        if prior is not None and prior.avatar_color is not None:
            return prior
        return Binding()
    companions: Counter = Counter()
    moves = 0
    for t in _informative(timeline):
        if t.before is None or t.after is None or t.action.action not in SIMPLE_MOVE_ACTIONS:
            continue
        # Loose delta (centroid displacement): the sprite's indicator part
        # mutates between frames, so exact-shape translation rarely holds.
        a = _cells_of(t.before, best_color)
        b = _cells_of(t.after, best_color)
        if not a or not b or abs(len(a) - len(b)) > max(2, len(a) // 4):
            continue
        ax = sum(x for x, _ in a) / len(a); ay = sum(y for _, y in a) / len(a)
        bx = sum(x for x, _ in b) / len(b); by = sum(y for _, y in b) / len(b)
        delta = (round(bx - ax), round(by - ay))
        if not _axis_jump(delta):
            continue
        moves += 1
        av = _cells_of(t.before, best_color)
        near = {
            (x + dx, y + dy)
            for x, y in av
            for dx in (-2, -1, 0, 1, 2)
            for dy in (-2, -1, 0, 1, 2)
        }
        changed = diff(t.before, t.after)
        dx, dy = delta
        for color in {old for _, _, old, _ in changed}:
            if color == best_color:
                continue
            # Cell-local: the same color may exist elsewhere on the board
            # (ls20's goal room shares the sprite's color), so whole-color
            # translation can never match — look for c-cells NEAR the avatar
            # that vanish and reappear shifted by the avatar's delta.
            gone = {
                (x, y) for x, y, old, new in changed if old == color and new != color
            }
            came = {
                (x, y) for x, y, old, new in changed if new == color and old != color
            }
            shifted = {(x + dx, y + dy) for x, y in gone}
            if (
                gone
                and gone & near
                and len(shifted & came) >= max(2, len(gone) // 2)
            ):
                companions[color] += 1
    extra = frozenset(
        c for c, n in companions.items() if moves >= 3 and n >= 0.6 * moves
    )
    if not extra and prior is not None and prior.avatar_color == best_color:
        extra = prior.avatar_extra  # companions are sticky too
    return Binding(avatar_color=best_color, avatar_extra=extra)


# ---------------------------------------------------------------------------
# Candidate generation


def candidates_from(t: Transition, binding: Binding) -> list[Rule]:
    """Rules consistent with this single transition (dedup'd later)."""
    if t.before is None or t.after is None or t.action.is_reset():
        return []
    changed = diff(t.before, t.after)
    if not changed:
        return []
    out: list[Rule] = []
    action = t.action.action
    moved_colors: dict[int, tuple[int, int]] = {}
    for color in {old for _, _, old, _ in changed} | {new for _, _, _, new in changed}:
        delta = translated(t.before, t.after, color)
        if delta is not None:
            moved_colors[color] = delta

    if action in SIMPLE_MOVE_ACTIONS and binding.avatar_color in moved_colors:
        delta = moved_colors[binding.avatar_color]
        # The color vacated cells actually became — maze floors are NOT the
        # global background (most-common is often the WALL color).
        floors = Counter(
            new
            for _, _, old, new in changed
            if old == binding.avatar_color and new != binding.avatar_color
        )
        floor = floors.most_common(1)[0][0] if floors else None
        out.append(MoveRule(deltas=((action, delta),), floor=floor))
        # Colors that vanished where the avatar landed -> consumable.
        av_after = _cells_of(t.after, binding.avatar_color)
        eaten = {
            old
            for x, y, old, new in changed
            if new == binding.avatar_color and old not in (binding.avatar_color,)
            and old != binding.bg(t.before) and (x, y) in av_after
        }
        # A color that moved with the same delta AND stood in the cells the
        # avatar entered -> push. (Same-delta alone votes coincidental
        # co-movers — a patrolling enemy — into "pushable".)
        av_before = _cells_of(t.before, binding.avatar_color)
        entered_cells = {
            (x + delta[0], y + delta[1]) for x, y in av_before
        } - av_before
        pushed = {
            c for c, d in moved_colors.items()
            if c != binding.avatar_color and d == delta
            and entered_cells & _cells_of(t.before, c)
        }
        if eaten:
            out.append(MoveRule(deltas=((action, delta),), consumes=frozenset(eaten)))
        if pushed:
            out.append(MoveRule(deltas=((action, delta),), pushable=frozenset(pushed)))

    if action == 6 and t.action.x is not None:
        cx, cy = t.action.x, t.action.y
        target = t.before[cy * 64 + cx]
        recolors = {(old, new) for _, _, old, new in changed}
        if len(recolors) == 1:
            old, new = next(iter(recolors))
            if old == target:
                n_changed = len(changed)
                target_cells = _cells_of(t.before, old)
                clicked_obj = next(
                    (
                        o
                        for o in components(
                            t.before, colors={old}, background=binding.bg(t.before)
                        )
                        if (cx, cy) in o.cells
                    ),
                    None,
                )
                if n_changed == 1 and (cx, cy) == next(iter({(x, y) for x, y, _, _ in changed})):
                    out.append(ClickRule(scope="cell", mapping=((old, new),)))
                if clicked_obj is not None and {(x, y) for x, y, _, _ in changed} == set(
                    clicked_obj.cells
                ):
                    out.append(ClickRule(scope="object", mapping=((old, new),)))
                if {(x, y) for x, y, _, _ in changed} == set(target_cells):
                    out.append(ClickRule(scope="color", mapping=((old, new),)))

    for color, delta in moved_colors.items():
        if color != binding.avatar_color and abs(delta[0]) + abs(delta[1]) >= 1:
            out.append(TickRule(color=color, delta=delta))

    if t.state_after == "GAME_OVER" and binding.avatar_color is not None:
        cells = avatar_cells(t.after, binding) or avatar_cells(t.before, binding)
        g = t.after
        adjacent = set()
        bg = binding.bg(g)
        for x, y in cells:
            for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                if 0 <= nx < 64 and 0 <= ny < 64:
                    c = g[ny * 64 + nx]
                    if c not in (bg, binding.avatar_color):
                        adjacent.add(c)
        if adjacent:
            out.append(HazardRule(colors=frozenset(adjacent)))
    return out


# ---------------------------------------------------------------------------
# Evaluation (backtest) and specialization


@dataclass
class Report:
    support: int = 0
    contradictions: int = 0
    matches_by_level: dict[int, int] = field(default_factory=lambda: defaultdict(int))
    misses_by_level: dict[int, int] = field(default_factory=lambda: defaultdict(int))
    counterexamples: list[Transition] = field(default_factory=list)

    def healthy_for(self, level: int, max_misses: int = 4) -> bool:
        return (
            self.support > 0
            and self.misses_by_level.get(level, 0) <= max_misses
        )


EVALUATE_WINDOW = 160


def _informative(timeline: Timeline) -> list[Transition]:
    """Bounded backtest set: the recent window plus every level-up-adjacent
    transition (they carry the goal evidence)."""
    ts = timeline.transitions
    if len(ts) <= EVALUATE_WINDOW:
        return list(ts)
    keep = ts[-EVALUATE_WINDOW:]
    extras = [t for t in ts[:-EVALUATE_WINDOW] if t.level_up]
    return extras + keep


def evaluate(
    rules: RuleSet,
    timeline: Timeline,
    binding: Binding,
    volatile: frozenset[int] = frozenset(),
    depleting: frozenset[int] = frozenset(),
) -> Report:
    """Replay recorded transitions through the interpreter and compare grids
    exactly outside the volatility mask (level-up frames excluded: the env
    swaps the board). Bounded by EVALUATE_WINDOW to keep re-induction
    affordable."""
    report = Report()
    for t in _informative(timeline):
        if (
            t.before is None
            or t.after is None
            or t.action.is_reset()
            or t.level_up
            or t.state_after == "GAME_OVER"
        ):
            continue
        predicted, _ = step(State(t.before), t.action.key(), rules, binding)
        if masked_eq(
            predicted.grid, t.after, volatile, depleting, keep=binding.avatar_color
        ):
            report.support += 1
            report.matches_by_level[t.level] += 1
        else:
            report.contradictions += 1
            report.misses_by_level[t.level] += 1
            if len(report.counterexamples) < 16:
                report.counterexamples.append(t)
    return report


def specialize(rule: MoveRule, counterexample: Transition, binding: Binding) -> list[MoveRule]:
    """Refine a move rule against one counterexample (guard addition only,
    depth is bounded by the caller)."""
    out: list[MoveRule] = []
    t = counterexample
    if t.before is None or t.after is None:
        return out
    delta = rule.delta_for(t.action.action)
    if delta is None:
        return out
    cells = avatar_cells(t.before, binding)
    if not cells:
        return out
    entered = {(x + delta[0], y + delta[1]) for x, y in cells} - cells
    entered_colors = {
        t.before[y * 64 + x]
        for x, y in entered
        if 0 <= x < 64 and 0 <= y < 64
    }
    moved = translated(t.before, t.after, binding.avatar_color or -1) is not None
    unchanged = t.before == t.after
    if not unchanged:
        av = binding.avatar_color
        unchanged = (
            av is not None
            and avatar_cells(t.before, binding) == avatar_cells(t.after, binding)
        )
    if unchanged and entered_colors:
        # Predicted a move but nothing happened: entered color blocks.
        for c in entered_colors - rule.blockers - {binding.bg(t.before)}:
            out.append(
                MoveRule(
                    deltas=rule.deltas,
                    blockers=rule.blockers | {c},
                    pushable=rule.pushable,
                    consumes=rule.consumes,
                    on_block=rule.on_block,
                    slide=rule.slide,
                )
            )
    elif moved and entered_colors - rule.consumes - {binding.bg(t.before)}:
        # Moved through a color we thought would block: mark consumable.
        for c in entered_colors - rule.consumes - {binding.bg(t.before)} - rule.blockers:
            out.append(
                MoveRule(
                    deltas=rule.deltas,
                    blockers=rule.blockers,
                    pushable=rule.pushable,
                    consumes=rule.consumes | {c},
                    on_block=rule.on_block,
                    slide=rule.slide,
                )
            )
    return out


# ---------------------------------------------------------------------------
# Goal inference


def infer_goal_candidates(timeline: Timeline, binding: Binding) -> list[Goal]:
    """Predicates true at every level-up and false at non-level-up states
    of the same level (negative examples are what make this sound). The
    consistent list is exposed so an oracle can break ties."""
    level_ups = [
        t
        for t in timeline.transitions
        if t.level_up and t.before is not None and t.after is not None
    ]
    if not level_ups:
        return []
    candidates: list[Goal] = []
    first = level_ups[0]
    bg = binding.bg(first.before)
    if binding.avatar_color is not None:
        cells = avatar_cells(first.before, binding)
        adjacent = set()
        for x, y in cells:
            for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                if 0 <= nx < 64 and 0 <= ny < 64:
                    c = first.before[ny * 64 + nx]
                    if c not in (bg, binding.avatar_color):
                        adjacent.add(c)
        candidates.extend(Goal("reach_color", c) for c in adjacent)
    before_counts = Counter(first.before)
    candidates.extend(
        Goal("clear_color", c)
        for c in before_counts
        if c not in (bg, binding.avatar_color) and before_counts[c] <= 4
    )
    # counter_eq: colors fully consumed over the course of the level.
    for seg in timeline.levels():
        if not seg.completed or not seg.transitions:
            continue
        start = seg.transitions[0].before
        end = seg.transitions[-1].after
        if start is None or end is None:
            continue
        start_counts = Counter(start)
        end_counts = Counter(end)
        for c in start_counts:
            if c in (bg, binding.avatar_color):
                continue
            gone = start_counts[c] - end_counts.get(c, 0)
            if gone > 0 and end_counts.get(c, 0) == 0:
                candidates.append(Goal("counter_eq", c, count=gone))
        break

    def consistent(goal: Goal) -> bool:
        # The state satisfying the predicate is unobservable (the board swaps
        # to the next level on completion), so soundness comes from the
        # NEGATIVE examples: the predicate must be false at every earlier
        # state of each level segment.
        for seg in timeline.levels():
            for t in seg.transitions[:-1]:
                if t.before is None:
                    continue
                if goal.kind in ("reach_color", "clear_color") and is_goal(
                    State(t.before), goal, binding
                ):
                    return False
        return True

    consistent_goals = [g for g in candidates if consistent(g)]
    return consistent_goals or (candidates[:1] if candidates else [])


def infer_goal(timeline: Timeline, binding: Binding) -> Goal | None:
    goals = infer_goal_candidates(timeline, binding)
    return goals[0] if goals else None


# ---------------------------------------------------------------------------
# Orchestration


@dataclass
class Model:
    binding: Binding
    rules: RuleSet
    report: Report
    goal: Goal | None
    volatile: frozenset[int] = frozenset()
    depleting: frozenset[int] = frozenset()

    def masked(self, g: Grid) -> Grid:
        return masked(
            g, self.volatile, self.depleting, keep=self.binding.avatar_color
        )

    def matches(self, a: Grid, b: Grid) -> bool:
        return masked_eq(
            a, b, self.volatile, self.depleting, keep=self.binding.avatar_color
        )

    def healthy_for(self, level: int) -> bool:
        return bool(self.rules) and self.report.healthy_for(level)


def induce(
    timeline: Timeline,
    max_specialize_rounds: int = 4,
    prior_binding: Binding | None = None,
) -> Model:
    """Full (re)induction over the timeline. Time cost is bounded by the
    caller's cadence; the binding is sticky across calls via prior_binding."""
    binding = rebind(timeline, prior=prior_binding)
    volatile = volatile_cells(timeline)
    depleting = depleting_colors(timeline)
    votes: Counter[Rule] = Counter()
    for t in _informative(timeline):
        for rule in candidates_from(t, binding):
            votes[rule] += 1

    move_deltas: dict[int, Counter] = defaultdict(Counter)
    consumes: set[int] = set()
    pushable: set[int] = set()
    floors: Counter = Counter()
    for rule, n in votes.items():
        if isinstance(rule, MoveRule):
            for action, delta in rule.deltas:
                move_deltas[action][delta] += n
            consumes |= set(rule.consumes)
            pushable |= set(rule.pushable)
            if rule.floor is not None:
                floors[rule.floor] += n
    rules: list[Rule] = []
    if move_deltas:
        assembled = []
        slide = False
        for action, counter in sorted(move_deltas.items()):
            dirs = Counter()
            for delta, n in counter.items():
                dirs[_direction(delta)] += n
            direction = dirs.most_common(1)[0][0]
            mags = Counter(
                abs(d[0]) + abs(d[1])
                for d, n in counter.items()
                if _direction(d) == direction
                for _ in range(n)
            )
            if len([m for m, n in mags.items() if n >= 2]) >= 2:
                # Same direction, varying distance: a slide (move until
                # blocked). Unit delta; the interpreter repeats the step.
                slide = True
                assembled.append((action, direction))
            else:
                mag = mags.most_common(1)[0][0]
                assembled.append(
                    (action, (direction[0] * mag, direction[1] * mag))
                )
        deltas = tuple(assembled)
        delta_map = dict(deltas)
        # Level-up transitions hide their consume event behind the board
        # swap (after = next level's grid), so mine the BEFORE grid: the
        # cell the avatar entered to complete a level must be enterable.
        for t in timeline.transitions:
            if not t.level_up or t.before is None:
                continue
            delta = delta_map.get(t.action.action)
            if delta is None or binding.avatar_color is None:
                continue
            cells = avatar_cells(t.before, binding)
            bg = binding.bg(t.before)
            for x, y in cells:
                nx, ny = x + delta[0], y + delta[1]
                if (nx, ny) not in cells and 0 <= nx < 64 and 0 <= ny < 64:
                    c = t.before[ny * 64 + nx]
                    if c not in (bg, binding.avatar_color):
                        consumes.add(c)
        # Blockers learned from CONFIRMED no-change moves (not only from
        # mispredictions: default-block semantics means a correct "blocked"
        # prediction never yields a counterexample, so specialization alone
        # would leave blockers empty and every wall an "untested assumption").
        blockers: Counter = Counter()
        # Regime guard: only count blocker evidence while the movement
        # system is demonstrably ALIVE (an energy-exhausted avatar no-ops
        # against everything, which would teach "the floor is a wall").
        informative = _informative(timeline)
        moved_recently: list[bool] = []
        window: list[bool] = []
        for t in informative:
            moved = (
                t.before is not None
                and t.after is not None
                and binding.avatar_color is not None
                and avatar_cells(t.before, binding) != avatar_cells(t.after, binding)
            )
            moved_recently.append(any(window[-4:]) or moved)
            window.append(moved)
        for idx, t in enumerate(informative):
            if (
                t.before is None
                or t.after is None
                or t.action.action not in delta_map
                or not moved_recently[idx]
            ):
                continue
            if masked(t.before, volatile, depleting) != masked(
                t.after, volatile, depleting
            ):
                continue  # something real changed: not a blocked move
            delta = delta_map[t.action.action]
            cells = avatar_cells(t.before, binding)
            bgc = binding.bg(t.before)
            for x, y in cells:
                nx, ny = x + delta[0], y + delta[1]
                if (nx, ny) not in cells and 0 <= nx < 64 and 0 <= ny < 64:
                    c = t.before[ny * 64 + nx]
                    if c in (bgc, binding.avatar_color):
                        continue
                    bx, by = nx + delta[0], ny + delta[1]
                    behind_free = (
                        0 <= bx < 64 and 0 <= by < 64
                        and t.before[by * 64 + bx] == bgc
                    )
                    if behind_free:
                        # The cell behind was free, so the color itself
                        # refused entry — true blocker evidence. A jammed
                        # push chain is NOT evidence against pushability.
                        blockers[c] += 1
        confirmed = frozenset(
            c for c, n in blockers.items()
            if n >= 2 and c not in consumes and c not in pushable
        )
        rules.append(
            MoveRule(
                deltas=deltas,
                blockers=confirmed,
                consumes=frozenset(consumes),
                pushable=frozenset(pushable),
                floor=floors.most_common(1)[0][0] if floors else None,
                slide=slide,
            )
        )
    click_votes = [
        (n, rule) for rule, n in votes.items() if isinstance(rule, ClickRule)
    ]
    if click_votes:
        # Prefer the most-supported scope; merge mappings of that scope.
        scope = max(
            ("cell", "object", "color"),
            key=lambda s: sum(n for n, r in click_votes if r.scope == s),
        )
        mapping: dict[int, int] = {}
        for n, r in sorted(click_votes, reverse=True, key=lambda p: p[0]):
            if r.scope == scope:
                for src, dst in r.mapping:
                    mapping.setdefault(src, dst)
        if mapping:
            rules.append(ClickRule(scope=scope, mapping=tuple(sorted(mapping.items()))))
    tick_by_color: dict[int, Counter] = defaultdict(Counter)
    n_change = sum(
        1
        for t in _informative(timeline)
        if t.before is not None and t.after is not None and t.before != t.after
        and not t.action.is_reset()
    )
    for rule, n in votes.items():
        if isinstance(rule, TickRule):
            if binding.avatar_color is None or rule.color != binding.avatar_color:
                tick_by_color[rule.color][rule.delta] += n
    for color, deltas in tick_by_color.items():
        delta, n = deltas.most_common(1)[0]
        # Majority delta only — a bouncing patroller votes both directions;
        # emitting both would cancel out. And a true ticker moves on most
        # transitions: occasionally-translated colors are pushed objects,
        # not passive dynamics.
        if n >= max(3, int(0.4 * n_change)):
            rules.append(TickRule(color=color, delta=delta))
    hazard_colors: Counter[int] = Counter()
    for rule, n in votes.items():
        if isinstance(rule, HazardRule):
            for c in rule.colors:
                hazard_colors[c] += n
    if hazard_colors:
        rules.append(
            HazardRule(colors=frozenset(c for c, n in hazard_colors.items() if n >= 1))
        )

    ruleset: RuleSet = tuple(rules)
    report = evaluate(ruleset, timeline, binding, volatile, depleting)
    for _ in range(max_specialize_rounds):
        if not report.counterexamples:
            break
        move = next((r for r in ruleset if isinstance(r, MoveRule)), None)
        if move is None:
            break
        improved = False
        for ce in report.counterexamples:
            for refined in specialize(move, ce, binding):
                trial = tuple(refined if r is move else r for r in ruleset)
                # Screen against the single counterexample before paying for
                # a full backtest.
                predicted, _ = step(State(ce.before), ce.action.key(), trial, binding)
                if masked(predicted.grid, volatile) != masked(ce.after, volatile):
                    continue
                trial_report = evaluate(trial, timeline, binding, volatile, depleting)
                if trial_report.contradictions < report.contradictions:
                    ruleset, report = trial, trial_report
                    improved = True
                    break
            if improved:
                break
        if not improved:
            break

    goal = infer_goal(timeline, binding)
    return Model(binding=binding, rules=ruleset, report=report, goal=goal, volatile=volatile, depleting=depleting)


In [ ]:
%%writefile /tmp/ouro2/plan.py
"""Search inside the induced model. Pure search: execution and
predicted-vs-observed checking belong to the director.

States are (grid bytes, counters) — exact and hashable. At the node cap
(default 20k) full-grid states cost tens of milliseconds, which buys the
simplicity of never reconstructing state from a factored encoding.
"""
from __future__ import annotations

import time
from collections import deque

from .grid import Grid, components, diff as _diff, most_common_color
from .rules import Binding, ClickRule, Goal, MoveRule, RuleSet, State, is_goal, step
from .timeline import ActionSpec

SIMPLE_ACTIONS = (1, 2, 3, 4, 5, 7)


def _actions(state: State, rules: RuleSet, binding: Binding,
             legal: tuple[int, ...]) -> list[tuple[int, int | None, int | None]]:
    out: list[tuple[int, int | None, int | None]] = []
    move = next((r for r in rules if isinstance(r, MoveRule)), None)
    if move is not None:
        for action, _ in move.deltas:
            if action in legal:
                out.append((action, None, None))
    click_sources: set[int] = set()
    for r in rules:
        if isinstance(r, ClickRule):
            click_sources |= {src for src, _ in r.mapping}
    if 6 in legal and click_sources:
        bg = binding.bg(state.grid)
        for obj in components(state.grid, colors=click_sources, background=bg)[:16]:
            out.append((6, obj.centroid[0], obj.centroid[1]))
    return out


def plan(
    state: State,
    rules: RuleSet,
    binding: Binding,
    goal: Goal,
    legal: tuple[int, ...] = (1, 2, 3, 4),
    node_cap: int = 20000,
    deadline_s: float = 2.0,
) -> list[ActionSpec] | None:
    """BFS to the first state satisfying the goal. Returns the action list,
    or None if unreachable within caps."""
    if is_goal(state, goal, binding):
        return []
    started = time.monotonic()
    start_key = (state.grid, state.counters)
    seen = {start_key}
    queue: deque[tuple[State, list[tuple[int, int | None, int | None]]]] = deque()
    queue.append((state, []))
    expanded = 0
    while queue:
        expanded += 1
        if expanded > node_cap:
            return None
        if expanded % 256 == 0 and time.monotonic() - started > deadline_s:
            return None
        current, path = queue.popleft()
        for action_key in _actions(current, rules, binding, legal):
            nxt, outcome = step(current, action_key, rules, binding)
            if nxt.status == "GAME_OVER" or outcome in ("blocked", "noop"):
                continue
            key = (nxt.grid, nxt.counters)
            if key in seen:
                continue
            seen.add(key)
            new_path = path + [action_key]
            if is_goal(nxt, goal, binding):
                return [
                    ActionSpec(a, x, y, source="plan", reason=f"{goal.kind}:{goal.color}")
                    for a, x, y in new_path
                ]
            queue.append((nxt, new_path))
    return None


def plan_frontier(
    state: State,
    rules: RuleSet,
    binding: Binding,
    seen: set[str],
    legal: tuple[int, ...] = (1, 2, 3, 4),
    node_cap: int = 20000,
    deadline_s: float = 1.0,
    max_len: int = 40,
    volatile: frozenset[int] = frozenset(),
    depleting: frozenset[int] = frozenset(),
    banned: set | None = None,
) -> list[ActionSpec] | None:
    """BFS inside the model to the nearest state NOT seen in reality yet.

    This is directed exploration: before any goal is known (level 0, the
    lab), a verified move-model plus wall knowledge turns exploration from
    a rotor-router walk into shortest-path frontier probing.
    """
    from .grid import masked_key

    started = time.monotonic()
    start_key = (state.grid, state.counters)
    seen_model = {start_key}
    queue: deque[tuple[State, list[tuple[int, int | None, int | None]]]] = deque()
    queue.append((state, []))
    expanded = 0
    avatar = binding.avatar_color
    bg = binding.bg(state.grid)
    move = next((r for r in rules if isinstance(r, MoveRule)), None)
    start_masked = _mask(state.grid, volatile, depleting, keep=avatar)
    # rank: 0 = interaction novelty, 1 = untested-block probe, 2 = reposition
    found: list[tuple[int, int, list[tuple[int, int | None, int | None]]]] = []
    stop_depth = max_len
    while queue:
        expanded += 1
        if expanded > node_cap:
            break
        if expanded % 256 == 0 and time.monotonic() - started > deadline_s:
            break
        current, path = queue.popleft()
        if len(path) >= min(stop_depth, max_len):
            continue
        for action_key in _actions(current, rules, binding, legal):
            nxt, outcome = step(current, action_key, rules, binding)
            if nxt.status == "GAME_OVER":
                continue
            if outcome in ("blocked", "noop"):
                # The model may be wrong about this block: a color it never
                # SAW block (absent from learned blockers) is an untested
                # assumption — the highest-information experiment when plain
                # novelty runs dry (a sokoban box was "blocked" from the
                # target cell only by default classification).
                if (
                    move is not None
                    and outcome == "blocked"
                    and avatar is not None
                ):
                    entered = _entered_colors(current, action_key, move, binding)
                    already = banned is not None and (
                        masked_key(
                            _mask(current.grid, volatile, depleting, keep=avatar),
                            frozenset(),
                        ),
                        action_key,
                    ) in banned
                    if entered and not (entered & move.blockers) and not already:
                        found.append((1, len(path) + 1, path + [action_key]))
                continue
            key = (nxt.grid, nxt.counters)
            if key in seen_model:
                continue
            seen_model.add(key)
            new_path = path + [action_key]
            if masked_key(_mask(nxt.grid, volatile, depleting, keep=avatar), frozenset()) not in seen:
                # Interactions (a non-avatar object changed, judged on the
                # MASKED grids so tickers don't pollute) outrank walking.
                interaction = any(
                    old not in (avatar, bg) or new not in (avatar, bg)
                    for _, _, old, new in _diff(start_masked, _mask(nxt.grid, volatile, depleting, keep=avatar))
                )
                found.append((0 if interaction else 2, len(new_path), new_path))
                if interaction:
                    stop_depth = len(new_path)
                elif stop_depth == max_len:
                    stop_depth = len(new_path) + 4
                if len(found) >= 16:
                    queue.clear()
                    break
                continue
            queue.append((nxt, new_path))
    if not found:
        return None
    found.sort(key=lambda f: (f[0], f[1]))
    best = found[0][2]
    return [
        ActionSpec(a, x, y, source="plan-frontier", reason="novel state")
        for a, x, y in best
    ]


def _mask(
    g: Grid,
    volatile: frozenset[int],
    depleting: frozenset[int] = frozenset(),
    keep: int | None = None,
) -> Grid:
    from .induce import masked

    return masked(g, volatile, depleting, keep=keep)


def _entered_colors(state: State, action_key, move: MoveRule, binding: Binding):
    from .rules import avatar_cells

    delta = move.delta_for(action_key[0])
    if delta is None:
        return frozenset()
    cells = avatar_cells(state.grid, binding)
    bgc = binding.bg(state.grid)
    out = set()
    for x, y in cells:
        nx, ny = x + delta[0], y + delta[1]
        if (nx, ny) not in cells and 0 <= nx < 64 and 0 <= ny < 64:
            c = state.grid[ny * 64 + nx]
            if c not in (bgc, binding.avatar_color):
                out.add(c)
    return frozenset(out)


def plan_greedy(
    state: State,
    rules: RuleSet,
    binding: Binding,
    goal: Goal,
    legal: tuple[int, ...] = (1, 2, 3, 4),
) -> ActionSpec | None:
    """One-step fallback: the action whose predicted state gets closest to
    the goal color without dying."""
    targets = [
        (i % 64, i // 64)
        for i, c in enumerate(state.grid)
        if c == goal.color
    ]
    if not targets:
        return None
    best: tuple[int, ActionSpec] | None = None
    for action_key in _actions(state, rules, binding, legal):
        nxt, outcome = step(state, action_key, rules, binding)
        if nxt.status == "GAME_OVER" or outcome in ("blocked", "noop"):
            continue
        from .rules import avatar_cells

        cells = avatar_cells(nxt.grid, binding)
        if not cells:
            continue
        dist = min(
            abs(x - tx) + abs(y - ty) for x, y in cells for tx, ty in targets
        )
        spec = ActionSpec(action_key[0], action_key[1], action_key[2],
                         source="plan-greedy", reason="distance")
        if best is None or dist < best[0]:
            best = (dist, spec)
    return best[1] if best else None


In [ ]:
%%writefile /tmp/ouro2/explore.py
"""The fallback explorer — the floor beneath the world-model path.

One coherent policy instead of V1's nine solvers: untried simple actions
first, then untried salient clicks, then the least-recently-used legal
action. No-op (state, action) pairs are banned permanently — wall memory
persists across level resets and plays (V1 wiped it and re-paid the cost
after every death).
"""
from __future__ import annotations

from .grid import Grid, components, grid_key
from .timeline import ActionSpec

SIMPLE_ACTIONS = (1, 2, 3, 4, 5, 7)


class Explorer:
    def __init__(self) -> None:
        self.noop_bans: set[tuple[str, tuple[int, int | None, int | None]]] = set()
        # Milk-the-productive-click: consecutive same-click changes
        self.last_click: tuple[int, int] | None = None
        self.last_click_streak = 0
        self.tried: dict[str, set[tuple[int, int | None, int | None]]] = {}
        # LRU is keyed per (state, action): per-state rotor-router sweeps
        # provably cover the reachable state graph, while a global LRU would
        # round-robin up/down/left/right into a net-zero loop.
        self.last_used: dict[tuple[str, tuple[int, int | None, int | None]], int] = {}
        self.clock = 0
        self.color_stats: dict[int, list[int]] = {}  # color -> [changes, tries]
        self.cell_stats: dict[tuple[int, int], list[int]] = {}  # (x,y) -> same

    def note_result(
        self,
        state_key: str,
        action: ActionSpec,
        changed: bool,
        grid: Grid | None = None,
        novel: bool = True,
    ) -> None:
        self.tried.setdefault(state_key, set()).add(action.key())
        if not changed and not action.is_reset():
            self.noop_bans.add((state_key, action.key()))
        if action.action == 6 and action.x is not None:
            # Milk only while the click produces NOVEL states: a changing-
            # but-revisiting board is an oscillating toggle (the cn04
            # reward-hack), not progress.
            productive = changed and novel
            if productive and self.last_click == (action.x, action.y):
                self.last_click_streak += 1
            elif productive:
                self.last_click = (action.x, action.y)
                self.last_click_streak = 1
            else:
                self.last_click = None
                self.last_click_streak = 0
        # Global click priors: most targets are inert in EVERY state, and
        # per-state bans alone re-pay the whole sweep after each board
        # change. Learn per-cell and per-color outcomes once, globally.
        if action.action == 6 and grid is not None and action.x is not None:
            color = grid[action.y * 64 + action.x]
            cs = self.color_stats.setdefault(color, [0, 0])
            cs[0] += 1 if changed else 0
            cs[1] += 1
            cell = self.cell_stats.setdefault((action.x, action.y), [0, 0])
            cell[0] += 1 if changed else 0
            cell[1] += 1

    def ban(self, state_key: str, action: ActionSpec) -> None:
        """Permanent ban (deaths): never repeat this exact mistake."""
        self.noop_bans.add((state_key, action.key()))
        self.tried.setdefault(state_key, set()).add(action.key())

    def _click_targets(self, g: Grid) -> list[tuple[int, int]]:
        """Centroids round-robined across colors: a game's live controls are
        often one color among many inert ones, and a size-ranked cap can
        exclude that color entirely (ft09's color-9 tiles hid behind 24
        larger inert objects)."""
        by_color: dict[int, list[tuple[int, int]]] = {}
        for o in components(g):
            by_color.setdefault(o.color, []).append(o.centroid)
        out: list[tuple[int, int]] = []
        round_idx = 0
        # Cover ALL components (cap well above real boards' counts): ft09
        # has 20 same-color tiles where only the central board ones respond —
        # a small cap starved the list before reaching them.
        while len(out) < 96:
            added = False
            for color in sorted(by_color):
                targets = by_color[color]
                if round_idx < len(targets):
                    out.append(targets[round_idx])
                    added = True
                    if len(out) >= 96:
                        break
            if not added:
                break
            round_idx += 1
        # Cell-precision targets for small pattern boards: paint/copy games
        # respond to individual CELLS, not object centroids (the audit's
        # "center-of-object only" blind spot). Enumerate cells of up to two
        # mid-sized objects.
        boards = [
            o for o in components(g) if 9 <= o.size <= 49 and o.width >= 3 and o.height >= 3
        ][:2]
        for o in boards:
            for x, y in sorted(o.cells):
                if len(out) >= 140:
                    break
                out.append((x, y))
        return out

    def _ranked_clicks(self, g: Grid) -> list[tuple[int, int]]:
        """Click targets ordered by learned promise: colors that have
        produced changes first (optimistic prior for the untried), then
        least-tried cells; cells proven inert twice are dropped."""
        scored = []
        fallback = []
        for x, y in self._click_targets(g):
            color = g[y * 64 + x]
            ch, tr = self.color_stats.get(color, (0, 0))
            cch, ctr = self.cell_stats.get((x, y), (0, 0))
            fallback.append((x, y))
            if ctr >= 2 and cch == 0:
                continue  # globally inert cell
            prior = (ch + 1) / (tr + 2)
            scored.append((-prior, ctr, x, y))
        if not scored:
            return fallback
        scored.sort()
        return [(x, y) for _, _, x, y in scored]

    def next(self, g: Grid, legal: list[int]) -> ActionSpec:
        self.clock += 1
        key = grid_key(g)
        tried = self.tried.get(key, set())
        # Milk a productive click: a click that keeps changing the board is
        # doing WORK (counters, fills, cycles) — repeat it until it stops
        # (V1's paired-control and large-click replay, generalized).
        if (
            6 in legal
            and self.last_click is not None
            and 1 <= self.last_click_streak < 24
        ):
            x, y = self.last_click
            if (key, (6, x, y)) not in self.noop_bans:
                self.last_used[(key, (6, x, y))] = self.clock
                return ActionSpec(6, x, y, source="explore", reason="milk click")
        for action in SIMPLE_ACTIONS:
            if action in legal:
                k = (action, None, None)
                if k not in tried and (key, k) not in self.noop_bans:
                    self.last_used[(key, k)] = self.clock
                    return ActionSpec(action, source="explore", reason="untried action")
        if 6 in legal:
            for x, y in self._ranked_clicks(g):
                k = (6, x, y)
                if k not in tried and (key, k) not in self.noop_bans:
                    self.last_used[(key, k)] = self.clock
                    return ActionSpec(6, x, y, source="explore", reason="untried click")
        candidates: list[tuple[int, tuple[int, int | None, int | None]]] = []
        for action in legal:
            if action == 0:
                continue
            if action == 6:
                for x, y in self._ranked_clicks(g):
                    k = (6, x, y)
                    if (key, k) not in self.noop_bans:
                        candidates.append((self.last_used.get((key, k), 0), k))
            else:
                k = (action, None, None)
                if (key, k) not in self.noop_bans:
                    candidates.append((self.last_used.get((key, k), 0), k))
        if not candidates:  # everything banned here: retry oldest anyway,
            # clicks included — a state with only banned candidates must not
            # become a graveyard the run spins in.
            for action in legal:
                if action == 0:
                    continue
                if action == 6:
                    for x, y in self._click_targets(g):
                        k = (6, x, y)
                        candidates.append((self.last_used.get((key, k), 0), k))
                else:
                    k = (action, None, None)
                    candidates.append((self.last_used.get((key, k), 0), k))
        if not candidates:
            return ActionSpec(0, source="explore", reason="no legal actions")
        candidates.sort()
        _, k = candidates[0]
        self.last_used[(key, k)] = self.clock
        return ActionSpec(k[0], k[1], k[2], source="explore", reason="lru sweep")


In [ ]:
%%writefile /tmp/ouro2/oracle.py
"""Optional LLM selector. The 4B never authors content: it picks one item
from a CPU-generated list of choices, as JSON, at temperature 0, thinking
off. Any failure — transport, timeout, malformed JSON, out-of-menu answer
— returns the CPU default. The 0-call mode (oracle=None) is the product;
this is a tie-breaker, not a dependency.
"""
from __future__ import annotations

import json
import re
import threading
import urllib.request

from .config import Config

_GENERATION_LOCK = threading.Lock()  # one shared model across game threads

SYSTEM = (
    "You analyze a grid puzzle game. Answer with JSON only: "
    '{"choice": "<one of the offered options, verbatim>"}'
)


class Oracle:
    def __init__(self, config: Config, transport=None) -> None:
        self.config = config
        self.transport = transport  # injectable for tests
        self.calls_used = 0
        self.failures = 0
        self._model = None
        self._tokenizer = None

    def select(self, kind: str, question: str, choices: list[str], default: str) -> str:
        if not choices or len(choices) == 1:
            return default
        if self.calls_used >= self.config.model_max_calls:
            return default
        prompt = (
            f"[{kind}] {question}\nOptions:\n"
            + "\n".join(f"- {c}" for c in choices)
            + '\nAnswer with JSON: {"choice": "..."}'
        )
        self.calls_used += 1
        try:
            with _GENERATION_LOCK:
                raw = (
                    self.transport(prompt)
                    if self.transport is not None
                    else self._complete(prompt)
                )
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            answer = json.loads(match.group(0)) if match else {}
            choice = answer.get("choice")
            if choice in choices:
                return choice
        except Exception:  # noqa: BLE001 — fail open to the CPU default
            pass
        self.failures += 1
        return default

    # -- transports ------------------------------------------------------
    def _complete(self, prompt: str) -> str:
        if self.config.model_backend == "ollama":
            return self._ollama(prompt)
        return self._transformers(prompt)

    def _ollama(self, prompt: str) -> str:
        payload = json.dumps(
            {
                "model": self.config.model_path or "qwen3.5:4b-mlx",
                "messages": [
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": prompt},
                ],
                "stream": False,
                "think": False,
                "format": "json",
                "options": {"temperature": 0, "num_predict": 160},
            }
        ).encode()
        req = urllib.request.Request(
            "http://127.0.0.1:11434/api/chat",
            data=payload,
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            body = json.loads(resp.read())
        return body.get("message", {}).get("content", "")

    def _transformers(self, prompt: str) -> str:
        if self._model is None:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer

            self._tokenizer = AutoTokenizer.from_pretrained(self.config.model_path)
            self._model = AutoModelForCausalLM.from_pretrained(
                self.config.model_path,
                torch_dtype=torch.bfloat16,
                device_map={"": "cuda:0"},
            )
        messages = [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt},
        ]
        text = self._tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = self._tokenizer(text, return_tensors="pt").to(self._model.device)
        out = self._model.generate(
            **inputs, max_new_tokens=160, do_sample=False,
            pad_token_id=self._tokenizer.eos_token_id,
        )
        return self._tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )


In [ ]:
%%writefile /tmp/ouro2/director.py
"""The game director: observes, decides, and accounts for every action.

``choose(view)`` is the single entry point per turn: it first records the
previous action's observed result into the timeline (the framework gives
us the new frame as the next turn's input), then picks the next action.
It must never raise — my_agent wraps it fail-open, but the director's own
last resort is the explorer.
"""
from __future__ import annotations

import time
from collections import Counter
from dataclasses import dataclass, field

from .config import Config
from .explore import Explorer
from .grid import Grid, grid_key, masked_key, to_grid
from .induce import Model, induce
from .plan import plan, plan_frontier
from .rules import MoveRule, State, step
from .timeline import RESET, ActionSpec, Timeline


@dataclass(frozen=True)
class FrameView:
    """Framework-independent view of a FrameData."""

    grid: Grid | None
    state: str  # NOT_PLAYED | NOT_FINISHED | WIN | GAME_OVER
    levels_completed: int
    win_levels: int
    available_actions: tuple[int, ...]
    full_reset: bool = False

    @classmethod
    def from_frame(cls, frame) -> "FrameView":  # FrameData duck-typed
        return cls(
            grid=to_grid(frame.frame),
            state=getattr(frame.state, "name", str(frame.state)),
            levels_completed=frame.levels_completed,
            win_levels=getattr(frame, "win_levels", 0) or 0,
            available_actions=tuple(
                getattr(a, "value", a) for a in (frame.available_actions or [])
            ),
            full_reset=bool(getattr(frame, "full_reset", False)),
        )


@dataclass
class Ledger:
    actions_used: int = 0
    noops: int = 0
    revisits: int = 0
    plan_steps: int = 0
    plan_aborts: int = 0
    probes: int = 0
    resets: int = 0
    speedrun_actions: int = 0
    think_time_s: float = 0.0


class Director:
    def __init__(self, config: Config | None = None, oracle=None, game_id: str = "") -> None:
        self.config = config or Config()
        self.oracle = oracle
        self.game_id = game_id
        self.timeline = Timeline()
        self.explorer = Explorer()
        self.ledger = Ledger()
        self.last_grid: Grid | None = None
        self.last_action: ActionSpec | None = None
        self.last_levels = 0
        self.last_state = "NOT_PLAYED"
        self.seen_keys: set[str] = set()
        self.wants_speedrun_flag = False
        # World-model path
        self.model: Model | None = None
        self.plan_queue: list[tuple[ActionSpec, Grid]] = []
        self.pending_prediction: Grid | None = None
        self.reinduce_pending = False
        self.induced_at = 0  # timeline length at last induction
        self.level_start_grid: Grid | None = None
        self.plan_retry_at = 0  # timeline length gate after a failed plan
        # Post-WIN speedrun replay (a new play can only raise the score)
        self.speedrun_queue: list[tuple[ActionSpec, Grid | None]] = []
        self.speedrun_done = False
        self.last_novel_at = 0  # timeline length when a new state last appeared
        self.last_stuck_reset_at = 0
        self.aborts_since_induce = 0
        self.model_paused_until = 0  # circuit breaker: timeline length gate
        self.breaker_trips: dict[int, int] = {}
        self.model_disabled_levels: set[int] = set()
        self.frontier_disabled_levels: set[int] = set()
        self.frontier_strikes: dict[int, int] = {}
        self.plan_queue_source = ""
        # Observed transition graph over masked keys (model-free navigation)
        self.graph_edges: dict[tuple[str, tuple], Counter] = {}
        self.lethal_edges: set[tuple[str, tuple]] = set()
        self.key_grids: dict[str, Grid] = {}  # representative grid per key
        # Cumulative identity masks: monotone unions, so mask jitter between
        # inductions cannot thrash the key epoch (every rebuild wipes
        # exploration memory — rebuilds must be rare and meaningful).
        self.mask_volatile: frozenset[int] = frozenset()
        self.mask_depleting: frozenset[int] = frozenset()
        self.mask_avatar: int | None = None
        self.consec_move_noops = 0  # all-moves-dead soft-lock detector

    def _key(self, g: Grid) -> str:
        from .induce import masked

        return masked_key(
            masked(g, self.mask_volatile, self.mask_depleting, keep=self.mask_avatar),
            frozenset(),
        )

    # -- framework hooks -------------------------------------------------
    def wants_speedrun(self) -> bool:
        return self.wants_speedrun_flag

    def on_win(self, view: FrameView, remaining_actions: int = 0) -> bool:
        """Called from is_done when the WIN frame arrives (before any further
        choose). Records the final transition and decides whether to spend
        remaining budget on a speedrun replay. Idempotent: _record consumes
        the pending action."""
        try:
            self._record(view)
            self.last_grid = view.grid or self.last_grid
            self.last_levels = view.levels_completed
            self.last_state = view.state
            self.wants_speedrun_flag = self._decide_speedrun(remaining_actions)
        except Exception:  # noqa: BLE001
            self.wants_speedrun_flag = False
        return self.wants_speedrun_flag

    def _decide_speedrun(self, remaining_actions: int) -> bool:
        """After WIN: re-earn the game in a fresh play with a COMPRESSED
        path — per level, a BFS plan inside the induced model if shorter,
        else the recorded actions. Score = max over plays, so this can only
        raise the score; per-step verification aborts back to live play on
        any divergence. Skipped when nothing compresses (a verbatim replay
        can only tie)."""
        if self.speedrun_done:
            return False
        win_play = next(
            (
                p
                for p in self.timeline.plays()
                if any(t.state_after == "WIN" for t in p)
            ),
            None,
        )
        if win_play is None:
            return False
        queue: list[tuple[ActionSpec, Grid | None]] = []
        recorded_total = 0
        compressed = False
        segments: dict[int, list] = {}
        for t in win_play:
            if not t.full_reset:
                segments.setdefault(t.level, []).append(t)
        model = self.model
        for level in sorted(segments):
            ts = segments[level]
            recorded_total += len(ts)
            start = ts[0].before
            planned = None
            if model is not None and model.goal is not None and start is not None:
                planned = plan(
                    State(start),
                    model.rules,
                    model.binding,
                    model.goal,
                    node_cap=self.config.node_cap,
                    deadline_s=2.0,
                )
            if planned and len(planned) < len(ts):
                compressed = True
                sim = State(start)
                for i, spec in enumerate(planned):
                    sim, _ = step(sim, spec.key(), model.rules, model.binding)
                    expected = None if i == len(planned) - 1 else sim.grid
                    queue.append(
                        (
                            ActionSpec(
                                spec.action, spec.x, spec.y,
                                source="speedrun", reason="planned replay",
                            ),
                            expected,  # level-up swaps the board: unverifiable
                        )
                    )
            else:
                for t in ts:
                    a = t.action
                    queue.append(
                        (
                            ActionSpec(a.action, a.x, a.y, source="speedrun",
                                       reason="recorded replay"),
                            t.after,
                        )
                    )
        needed = len(queue) + 1  # +1 for the RESET opening the new play
        if not queue or not compressed or remaining_actions < needed * 1.15:
            return False
        self.speedrun_queue = queue
        self.speedrun_done = True
        return True

    def choose(self, view: FrameView) -> ActionSpec:
        started = time.monotonic()
        try:
            self._record(view)
            action = self._decide(view)
        except Exception:  # noqa: BLE001 — never abort the run
            action = self._explore(view)
        self.last_action = action
        if view.grid is not None:
            self.last_grid = view.grid
        self.last_levels = view.levels_completed
        self.last_state = view.state
        self.ledger.actions_used += 1
        if action.is_reset():
            self.ledger.resets += 1
        self.ledger.think_time_s += time.monotonic() - started
        return action

    def summary(self) -> dict:
        return {
            "game_id": self.game_id,
            "actions": self.ledger.actions_used,
            "noops": self.ledger.noops,
            "revisits": self.ledger.revisits,
            "plan_steps": self.ledger.plan_steps,
            "plan_aborts": self.ledger.plan_aborts,
            "probes": self.ledger.probes,
            "resets": self.ledger.resets,
            "speedrun_actions": self.ledger.speedrun_actions,
            "think_time_s": round(self.ledger.think_time_s, 3),
            "transitions": len(self.timeline),
            "levels_completed": self.last_levels,
        }

    # -- internals -------------------------------------------------------
    def _record(self, view: FrameView) -> None:
        if self.last_action is None:
            return
        action, self.last_action = self.last_action, None  # consume: idempotent
        self.timeline.append(
            before=self.last_grid,
            action=action,
            after=view.grid,
            state_after=view.state,
            levels_before=self.last_levels,
            levels_after=view.levels_completed,
            full_reset=view.full_reset,
        )
        if view.grid is not None and self.last_grid is not None:
            changed = view.grid != self.last_grid
            # For LEARNING, "changed" means changed outside the masks: a
            # click that only ticks an attempt counter is a no-op in every
            # way that matters.
            if changed and self.model is not None:
                changed = not self.model.matches(view.grid, self.last_grid)
            if not changed and not action.is_reset():
                self.ledger.noops += 1
            if action.action in (1, 2, 3, 4):
                self.consec_move_noops = 0 if changed else self.consec_move_noops + 1
            elif action.is_reset():
                self.consec_move_noops = 0
            key = self._key(view.grid)
            novel = key not in self.seen_keys
            if not novel:
                self.ledger.revisits += 1
            else:
                self.last_novel_at = len(self.timeline)
            self.seen_keys.add(key)
            self.key_grids.setdefault(key, view.grid)
            self.explorer.note_result(
                self._key(self.last_grid), action, changed,
                grid=self.last_grid, novel=novel,
            )
            if not action.is_reset():
                edge = (self._key(self.last_grid), action.key())
                self.graph_edges.setdefault(edge, Counter())[key] += 1
        # Verify the committed step (plan or speedrun): reality outranks
        # the model/recording.
        if self.pending_prediction is not None:
            mismatch = view.grid is not None and view.grid != self.pending_prediction
            if mismatch and self.model is not None:
                # Union-masked equality: absorbs HUD churn including freshly
                # DRAINED depleting cells (stale bar vs revealed floor); any
                # remaining difference is a real misprediction.
                mismatch = not self.model.matches(view.grid, self.pending_prediction)
            if mismatch:
                self.plan_queue.clear()
                self.speedrun_queue.clear()
                self.wants_speedrun_flag = False
                self.ledger.plan_aborts += 1
                self.reinduce_pending = True
                self.aborts_since_induce += 1
                if self.plan_queue_source == "plan-frontier":
                    # Three strikes per level: an abort feeds the model its
                    # counterexample (abort -> revise -> retry), but a model
                    # that keeps mispredicting loses frontier control until
                    # the next level.
                    level = view.levels_completed
                    self.frontier_strikes[level] = (
                        self.frontier_strikes.get(level, 0) + 1
                    )
                    if self.frontier_strikes[level] >= 3:
                        self.frontier_disabled_levels.add(level)
                if self.aborts_since_induce >= 6:
                    # Circuit breaker: a chronically mispredicting model must
                    # not keep steering (strict additivity) — floor only for
                    # a long window; two trips on one level disable the model
                    # path for that level entirely.
                    self.model_paused_until = len(self.timeline) + 64
                    self.aborts_since_induce = 0
                    level = view.levels_completed
                    self.breaker_trips[level] = self.breaker_trips.get(level, 0) + 1
            else:
                self.ledger.plan_steps += 1
            self.pending_prediction = None
        # Autopsy: never repeat the exact action that ended the game.
        if view.state == "GAME_OVER" and self.last_grid is not None:
            self.explorer.ban(self._key(self.last_grid), action)
            self.lethal_edges.add((self._key(self.last_grid), action.key()))
        # Track the level-start grid for consumed-counter reconstruction.
        if view.levels_completed != self.last_levels or view.full_reset or (
            action.is_reset() and view.grid is not None
        ):
            self.level_start_grid = view.grid
            self.plan_queue.clear()
            self.pending_prediction = None

    def _decide(self, view: FrameView) -> ActionSpec:
        if view.state == "NOT_PLAYED":
            return RESET
        if view.state == "GAME_OVER":
            self.plan_queue.clear()
            self.speedrun_queue.clear()
            self.wants_speedrun_flag = False
            self.pending_prediction = None
            self.reinduce_pending = True
            return RESET
        if view.state == "WIN":
            if self.wants_speedrun_flag and self.speedrun_queue:
                return RESET  # full reset at WIN: starts the replay play
            self.wants_speedrun_flag = False
            return RESET
        if view.grid is None:
            return self._explore(view)
        # Soft-lock escape: every recent move no-oped (ls20-class energy
        # exhaustion) — a level reset refills the resource; one action.
        if self.consec_move_noops >= 4 and self.timeline.current_level_transitions():
            self.consec_move_noops = 0
            self.reinduce_pending = True
            return ActionSpec(0, source="director", reason="soft-lock: moves dead")
        # Stuck escape: no novel state for a long stretch usually means an
        # irreversible dead position (cornered sokoban box) — a level reset
        # is the only exit, and it costs one action.
        if (
            len(self.timeline) - self.last_novel_at >= 96
            and len(self.timeline) - self.last_stuck_reset_at >= 96
            and self.timeline.current_level_transitions()
        ):
            self.last_stuck_reset_at = len(self.timeline)
            self.reinduce_pending = True
            return ActionSpec(0, source="director", reason="stuck: level reset")
        if self.speedrun_queue:
            action, expected = self.speedrun_queue.pop(0)
            self.pending_prediction = expected
            self.ledger.speedrun_actions += 1
            return action
        if self.plan_queue:
            action, predicted = self.plan_queue.pop(0)
            self.pending_prediction = predicted
            return action
        self._maybe_reinduce()
        planned = self._try_plan(view)
        if planned is not None:
            return planned
        return self._explore(view)

    def _maybe_reinduce(self) -> None:
        if self.ledger.think_time_s > self.config.time_budget_s:
            return  # over budget: degrade to the explorer floor
        grown = len(self.timeline) - self.induced_at
        if self.model is None or self.reinduce_pending or grown >= 16:
            if len(self.timeline) >= 4:
                self.model = induce(
                    self.timeline,
                    prior_binding=self.model.binding if self.model else None,
                )
                self.induced_at = len(self.timeline)
                self.reinduce_pending = False
                self.plan_retry_at = 0  # new evidence: planning may retry
                new_vol = self.mask_volatile | self.model.volatile
                new_dep = self.mask_depleting | self.model.depleting
                new_avatar = self.model.binding.avatar_color or self.mask_avatar
                if (
                    new_vol != self.mask_volatile
                    or new_dep != self.mask_depleting
                    or new_avatar != self.mask_avatar
                ):
                    self.mask_volatile = new_vol
                    self.mask_depleting = new_dep
                    self.mask_avatar = new_avatar
                    self._rebuild_keys()
                self._oracle_goal_tiebreak()

    def _rebuild_keys(self) -> None:
        """The state-identity function just changed (new volatility mask or
        avatar binding): every derived key set would fragment across epochs.
        The timeline is ground truth — rebuild them all under the new mask."""
        self.seen_keys.clear()
        self.graph_edges.clear()
        self.lethal_edges.clear()
        self.key_grids.clear()
        ex = self.explorer
        ex.tried.clear()
        ex.noop_bans.clear()
        ex.last_used.clear()
        for t in self.timeline.transitions:
            if t.after is not None:
                k_after = self._key(t.after)
                self.seen_keys.add(k_after)
                self.key_grids.setdefault(k_after, t.after)
            if t.before is None:
                continue
            key = self._key(t.before)
            ex.clock += 1
            if not t.action.is_reset():
                ex.last_used[(key, t.action.key())] = ex.clock
            changed = t.after is not None and t.after != t.before
            ex.note_result(key, t.action, changed, grid=t.before)
            if t.after is not None and not t.action.is_reset():
                self.graph_edges.setdefault(
                    (key, t.action.key()), Counter()
                )[self._key(t.after)] += 1
            if t.state_after == "GAME_OVER":
                ex.ban(key, t.action)
                self.lethal_edges.add((key, t.action.key()))

    def _oracle_goal_tiebreak(self) -> None:
        """When several goal predicates survive the negative examples, let
        the oracle pick; its answer is one of OUR candidates or ignored."""
        if self.oracle is None or self.model is None:
            return
        from .induce import infer_goal_candidates
        from .rules import Goal

        goals = infer_goal_candidates(self.timeline, self.model.binding)
        if len(goals) <= 1:
            return
        render = {f"{g.kind} color={g.color} n={g.count}": g for g in goals}
        choices = list(render)
        picked = self.oracle.select(
            "GOAL_SELECT",
            "Which condition most plausibly completes a level of this game?",
            choices,
            default=choices[0],
        )
        self.model.goal = render.get(picked, goals[0])

    def _frontier_eligible(self, model: Model, view: FrameView) -> bool:
        """Frontier probing needs only a usable move-model: misses may run
        higher than the exploit gate (probes are cheap and per-step
        verified — an abort just returns to the explorer)."""
        # Schema's bar: frontier probing (which steers real actions on pure
        # curiosity) only from a model with a CLEAN backtest for this level.
        # Goal-directed exploit planning keeps its small tolerance elsewhere.
        from .rules import TickRule

        return (
            any(isinstance(r, MoveRule) for r in model.rules)
            # v2.0 restriction: no frontier probing in ticker games — a
            # patroller's phase leaks into model-novelty in ways the masks
            # cannot fully hide, and the floor handles these games better.
            and not any(isinstance(r, TickRule) for r in model.rules)
            and model.binding.avatar_color is not None
            and model.report.support >= 12
            # Miss RATE, not zero misses: real games carry residual events
            # (ls20's terrain transforms) a mostly-right model mispredicts;
            # cheap per-step-verified probes + the one-strike policy bound
            # the cost of being wrong.
            and (
                model.report.misses_by_level.get(view.levels_completed, 0)
                <= 0.2 * max(1, model.report.support)
            )
        )

    def _consumed_counters(self, current: Grid) -> tuple[tuple[int, int], ...]:
        if self.level_start_grid is None:
            return ()
        start = Counter(self.level_start_grid)
        now = Counter(current)
        consumed = {
            c: start[c] - now.get(c, 0)
            for c in start
            if start[c] - now.get(c, 0) > 0
        }
        return tuple(sorted(consumed.items()))

    def _try_plan(self, view: FrameView) -> ActionSpec | None:
        model = self.model
        attempted = self.ledger.plan_steps + self.ledger.plan_aborts
        failing = attempted >= 6 and self.ledger.plan_aborts / attempted > 0.5
        if (
            model is None
            or len(self.timeline) < self.plan_retry_at
            or len(self.timeline) < self.model_paused_until
            or failing
            or self.ledger.think_time_s > self.config.time_budget_s
        ):
            return None
        state = State(view.grid, counters=self._consumed_counters(view.grid))
        legal = tuple(a for a in view.available_actions if a != 0) or (1, 2, 3, 4)
        actions = None
        if model.goal is not None and model.healthy_for(view.levels_completed):
            actions = plan(
                state,
                model.rules,
                model.binding,
                model.goal,
                legal=legal,
                node_cap=self.config.node_cap,
                deadline_s=2.0,
            )
        if (
            not actions
            and view.levels_completed not in self.frontier_disabled_levels
            and self._frontier_eligible(model, view)
        ):
            # No goal yet (level 0 is the lab) or goal unreachable: use the
            # verified move-model for shortest-path novelty probing instead
            # of rotor-router wandering.
            actions = plan_frontier(
                state,
                model.rules,
                model.binding,
                self.seen_keys,
                legal=legal,
                node_cap=self.config.node_cap,
                deadline_s=1.0,
                volatile=self.mask_volatile,
                depleting=self.mask_depleting,
                banned=self.explorer.noop_bans,
            )
            if actions:
                self.ledger.probes += len(actions)
        if not actions:
            # Cool down: don't re-pay BFS cost until new evidence arrives.
            self.plan_retry_at = len(self.timeline) + 8
            return None
        # Precompute per-step predicted grids for the executor.
        queue: list[tuple[ActionSpec, Grid]] = []
        sim = state
        for spec in actions:
            sim, _ = step(sim, spec.key(), model.rules, model.binding)
            queue.append((spec, sim.grid))
        self.plan_queue = queue
        self.plan_queue_source = actions[0].source if actions else ""
        action, predicted = self.plan_queue.pop(0)
        self.pending_prediction = predicted
        return action

    def _explore(self, view: FrameView) -> ActionSpec:
        if view.grid is None:
            return RESET
        legal = [a for a in view.available_actions if a != 0] or [1, 2, 3, 4]
        action = self.explorer.next(view.grid, list(legal))
        if action.reason == "lru sweep":
            # Nothing untried here: walk the OBSERVED graph to the nearest
            # state that still has untried actions (model-free, so it works
            # even when the induced model is unclean). Self-correcting: each
            # step re-runs this.
            routed = self._graph_frontier_step(view, legal)
            if routed is not None:
                return routed
        return action

    def _graph_frontier_step(self, view: FrameView, legal: list[int]) -> ActionSpec | None:
        from collections import deque

        start = self._key(view.grid)
        adjacency: dict[str, list] = {}
        for (from_key, action_key), targets in self.graph_edges.items():
            if (from_key, action_key) in self.lethal_edges:
                continue  # an edge that EVER killed is off-limits to routing
            adjacency.setdefault(from_key, []).append(
                (action_key, targets.most_common(1)[0][0])
            )
        seen = {start}
        queue = deque([(start, None)])  # (key, first action on path)
        expanded = 0
        while queue:
            expanded += 1
            if expanded > 600:
                return None
            key, first = queue.popleft()
            if key != start:
                tried = self.explorer.tried.get(key, set())
                for a in legal:
                    if a == 6:
                        grid = self.key_grids.get(key)
                        if grid is None:
                            continue
                        has_untried_click = any(
                            (6, x, y) not in tried
                            and (key, (6, x, y)) not in self.explorer.noop_bans
                            for x, y in self.explorer._ranked_clicks(grid)[:16]
                        )
                        if has_untried_click and first is not None:
                            return ActionSpec(
                                first[0], first[1], first[2],
                                source="graph-frontier", reason="route to untried click",
                            )
                        continue
                    k = (a, None, None)
                    if k not in tried and (key, k) not in self.explorer.noop_bans:
                        if first is None:
                            return None
                        return ActionSpec(
                            first[0], first[1], first[2],
                            source="graph-frontier", reason="route to untried",
                        )
            for action_key, nxt in adjacency.get(key, ()):
                if nxt in seen:
                    continue
                seen.add(nxt)
                queue.append((nxt, first if first is not None else action_key))
        return None


In [ ]:
%%writefile /tmp/ouro2/holdout.py
"""Holdout split (DEV/TEST/QUARANTINE) — fold literals identical to V1's.

The public leaderboard reruns on hidden games, so local play on the 25
public games is only an honest LB proxy if some games are held out of all
tuning. This module is the ONE sanctioned home for game-id literals; the
overfit lint enforces that no other module contains them.
"""
from __future__ import annotations

from dataclasses import dataclass

DEV_GAMES = frozenset(
    {
        "ft09", "m0r0", "sp80", "s5i5", "ls20", "lp85", "cn04",
        "tr87", "sb26", "sk48", "bp35", "r11l", "tu93",
    }
)

# LB proxy: read to report, never to tune.
TEST_GAMES = frozenset(
    {"vc33", "lf52", "su15", "sc25", "g50t", "wa30", "ka59", "dc22", "tn36"}
)

# Untouched until a declared final calibration before submission.
QUARANTINE_GAMES = frozenset({"ar25", "re86", "cd82"})

ALL_PUBLIC_GAMES = DEV_GAMES | TEST_GAMES | QUARANTINE_GAMES

FOLDS = {"dev": DEV_GAMES, "test": TEST_GAMES, "quarantine": QUARANTINE_GAMES}


def normalize_game_id(game_id: str) -> str:
    return game_id.split("-")[0].strip().lower()


@dataclass(frozen=True)
class GateResult:
    ok: bool
    reasons: tuple[str, ...]


def gate(test_results: dict, baseline: dict, eps: float = 0.005) -> GateResult:
    """Block when the TEST fold regresses vs the ratcheted baseline.

    ``test_results``/``baseline``: {"score": float, "levels": {game: int}}.
    """
    reasons = []
    base_levels = baseline.get("levels", {})
    got_levels = test_results.get("levels", {})
    for game, base in base_levels.items():
        if got_levels.get(game, 0) < base:
            reasons.append(
                f"TEST regression: {game} levels {got_levels.get(game, 0)} < {base}"
            )
    if test_results.get("score", 0.0) < baseline.get("score", 0.0) - eps:
        reasons.append(
            f"TEST score {test_results.get('score', 0.0):.4f} < "
            f"baseline {baseline.get('score', 0.0):.4f} - {eps}"
        )
    return GateResult(ok=not reasons, reasons=tuple(reasons))


In [ ]:
%%writefile /tmp/ouro2/__init__.py
"""ouro2 — from-scratch minimal ARC-AGI-3 Kaggle agent.

CPU induces mechanic rules from exact transition diffs; rules are data run
by a fixed interpreter; an optional LLM answers multiple-choice questions;
planning happens inside the induced model before real actions are spent.
"""
from .config import Config
from .timeline import ActionSpec, Timeline, Transition

__all__ = ["ActionSpec", "Config", "Timeline", "Transition"]


In [ ]:
%%writefile /tmp/my_agent.py
"""Kaggle ARC-AGI-3 submission agent (V2).

Thin adapter over ouro2.Director; copied into the ARC-AGI-3-Agents
framework by the generated notebook. All strategy lives in ouro2 so it is
testable without the competition framework installed.
"""
from __future__ import annotations

import os
from typing import Any

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent

from ouro2.config import Config
from ouro2.director import Director, FrameView
from ouro2.timeline import ActionSpec


def _build_oracle(config: Config):
    if config.disable_model:
        return None
    from ouro2.oracle import Oracle

    return Oracle(config)


class MyAgent(Agent):
    MAX_ACTIONS = int(os.getenv("OURO2_MAX_ACTIONS", "320"))

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        config = Config.from_env()
        self.director = Director(
            config=config,
            oracle=_build_oracle(config),
            game_id=str(getattr(self, "game_id", "unknown")),
        )

    @property
    def name(self) -> str:
        return f"{super().name}.ouro2.{self.MAX_ACTIONS}"

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        if latest_frame.state is not GameState.WIN:
            return False
        remaining = self.MAX_ACTIONS - self.action_counter
        try:
            speedrun = self.director.on_win(
                FrameView.from_frame(latest_frame), remaining_actions=remaining
            )
        except Exception:  # noqa: BLE001
            speedrun = False
        return not (remaining > 0 and speedrun)

    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        try:
            spec = self.director.choose(FrameView.from_frame(latest_frame))
        except Exception:  # noqa: BLE001 — a raised exception zeroes the run
            spec = ActionSpec(0, source="failsafe", reason="director error")
        return self._to_game_action(spec)

    def cleanup(self, scorecard: Any | None = None) -> None:
        super().cleanup(scorecard)
        try:
            print(f"[ouro2] {self.director.summary()}")
        except Exception:  # noqa: BLE001
            pass

    def _to_game_action(self, spec: ActionSpec) -> GameAction:
        if spec.is_reset():
            action = GameAction.RESET
        else:
            action = getattr(GameAction, f"ACTION{spec.action}")
        if spec.action == 6:
            action.set_data({"x": spec.x, "y": spec.y})
        action.reasoning = {"source": spec.source, "reason": spec.reason or spec.source}
        return action


In [ ]:

import os, shutil, subprocess, sys, time, urllib.request

os.environ.setdefault("MPLBACKEND", "agg")
os.environ.setdefault("OURO2_MAX_ACTIONS", "320")
os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001")
os.environ.setdefault("OURO2_DISABLE_MODEL", "0")
os.environ.setdefault("OURO2_MODEL_BACKEND", "transformers")
os.environ.setdefault("OURO2_MODEL_PATH", "/kaggle/input/models/kinwochan/qwen-3-5-4b/transformers/qwen-3-5-4b/1")
rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

def gateway_up(retries, delay):
    for _ in range(retries):
        try:
            urllib.request.urlopen("http://gateway:8001/api/games", timeout=5)
            return True
        except Exception:
            time.sleep(delay)
    return False

up = gateway_up(120, 5) if rerun else gateway_up(3, 1)
run_agent = rerun or up
print(f"rerun={rerun} gateway={up} -> {'arc-agent' if run_agent else 'dummy-submission'}")

if run_agent:
    src = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents"
    dst = "/kaggle/working/ARC-AGI-3-Agents"
    if not os.path.isdir(dst):
        shutil.copytree(src, dst)
    shutil.copy("/tmp/my_agent.py", f"{dst}/agents/templates/my_agent.py")
    if os.path.isdir(f"{dst}/ouro2"):
        shutil.rmtree(f"{dst}/ouro2")
    shutil.copytree("/tmp/ouro2", f"{dst}/ouro2")
    with open(f"{dst}/agents/__init__.py", "w") as fh:
        fh.write(
            "from .agent import Agent\n"
            "from .swarm import Swarm\n"
            "from .templates.my_agent import MyAgent\n"
            "AVAILABLE_AGENTS = {'myagent': MyAgent}\n"
        )
    with open(f"{dst}/.env", "w") as fh:
        fh.write("SCHEME=http\nHOST=gateway\nPORT=8001\n"
                 "ARC_API_KEY=test-key-123\nOPERATION_MODE=online\n")
    subprocess.run([sys.executable, "main.py", "--agent", "myagent"], cwd=dst, check=False)
else:
    import pandas as pd

    pd.DataFrame(
        [["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    ).to_parquet("/kaggle/working/submission.parquet")
    print("wrote dummy submission.parquet")

    model_path = os.environ.get("OURO2_MODEL_PATH", "")
    if model_path:
        # Save-run model smoke: the agent only runs in a competition rerun,
        # so this is the one chance to prove the attached model loads on the
        # rerun image before a scored run bets on it (the V1 V7/V8 zeroes
        # were exactly this failure). Fail-open: a failure here is loud in
        # the log but never fails the notebook.
        import traceback

        print(f"model-smoke: path={model_path} isdir={os.path.isdir(model_path)}")
        if not os.path.isdir(model_path):
            for root, dirs, _ in os.walk("/kaggle/input"):
                if root.count("/") <= 6:
                    print("  " + root)
                else:
                    dirs[:] = []
        try:
            t0 = time.time()
            sys.path.insert(0, "/tmp")
            from ouro2.config import Config
            from ouro2.oracle import Oracle

            raw = Oracle(Config.from_env())._transformers(
                'Reply with JSON: {"choice": "alpha"}'
            )
            print(f"model-smoke: OK {time.time() - t0:.1f}s raw={raw[:200]!r}")
        except Exception:
            traceback.print_exc()
            print("model-smoke: FAILED (a rerun would fail open to CPU defaults)")
